### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.8-27b", max_tokens=800)
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='It is a common misconception that parrots "talk" in the same way humans do. In reality, parrots **mimic** sounds, rather than truly understanding the semantic meaning of words in most cases. However, the reasons behind this behavior are rooted in their biology, social structure, and evolutionary history.\n\nHere’s a breakdown of why parrots mimic human speech and other sounds:\n\n### 1. Social Bonding\nParrots are highly social animals that live in flocks in the wild. Communication is central to their social structure. In captivity, your family or household becomes their "flock." By mimicking the sounds they hear most often—especially human speech—they are attempting to:\n- **Bond** with their human caretakers.\n- **Integrate** themselves into the social group.\n- **Gain attention** and interaction, which is essential for their mental well-being.\n\n### 2. Natural Vocal Mimicry\nParrots have a unique anatomical ability to produce a wide range of sounds. Their syrinx 

In [5]:
from langchain.tools import tool

@tool

def get_weather(location:str)-> str:
    """Get The weather at location"""
    return f"It's sunny in {location}"

model_with_tool = model.bind_tools([get_weather])

In [7]:
response = model_with_tool.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'jz12eaae7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.069380752, 'completion_tokens_details': None, 'prompt_time': 0.022000406, 'prompt_tokens_details': None, 'queue_time': 0.009289087, 'total_time': 0.091381158}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0ba36-0213-7451-8bc1-10c20261fbc1-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'jz12eaae7', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}
Tool: get_weather
Args: {'location': 'Boston'}


In [8]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's currently sunny in Boston. ☀️


In [14]:
import requests
from bs4 import BeautifulSoup
from langchain.tools import tool

@tool
def read_website(url: str) -> str:
    """Fetches and extracts the readable text content from a given website URL."""
    try:
        # Use a standard user-agent to prevent getting blocked by basic bot protection
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Parse the HTML
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Remove noisy elements like scripts, styles, and navigation
        for element in soup(['script', 'style', 'nav', 'footer', 'header']):
            element.extract()
            
        # Extract clean text
        text = soup.get_text(separator=' ', strip=True)
        
        # Limit the output to the first 3000 characters to fit context limits
        return text[:3000]
        
    except Exception as e:
        return f"Failed to fetch website content: {str(e)}"

In [19]:
# Call the tool directly with a test URL
scraped_text = read_website.invoke({"url": "https://en.wikipedia.org/wiki/ML"})

print(scraped_text)

ML - Wikipedia Jump to content From Wikipedia, the free encyclopedia Look up ML , Ml , mL , ml , .ml , ml. , Mℓ , or mℓ in Wiktionary, the free dictionary. ML or ml may refer to: Computing and mathematics [ edit ] ML (programming language) , a general-purpose functional programming language .ml , the top-level Internet domain for Mali Machine learning , a field of artificial intelligence Markup language , a system for annotating a document Mathematical Logic, a variation of Quine's system New Foundations Module-Lattice cryptography: ML-DSA, the Module-Lattice-Based Digital Signature Standard for post-quantum cryptography ML-KEM, the Module-Lattice-Based Key-Encapsulation Mechanism Standard for post-quantum cryptography MultiLevel Recording , to increase the storage capacity of optical discs Businesses [ edit ] Malaysia-Singapore Airlines , IATA code ML until 1972 M Lhuillier , a non-banking financial services company in the Philippines Merrill Lynch , the wealth management division of 